# Downstream analysis of decoded behaviour

Everything in this notebook reads the files a finished prediction run leaves
behind: `predictions.parquet` decoded into **bouts**, the per-movie
`tracks.parquet`, and the dense feature table. Point `PROJECT` at your `config.yaml` and run top to
bottom.

Sections:

1. **Mode switching** - the transition graph, jump matrix, dwell-time
   survival curves and the fitted Markov chain, drawn from the same numbers
   `transitions.json` reports.
2. **Behavioural tracks** - ethograms and trajectories coloured by the
   predicted behaviour.
3. **Behavioural motifs** - the keypoint-MoSeq trajectory-plot idiom per
   class (onset-aligned median pose trajectory), beside an instantaneous
   body-frame posture density.
4. **Feature distributions** - what the kinematic and waveform features look
   like inside each predicted behaviour.

`EXCLUDE_CLASSES` in the first cell drops classes (an "unclear" bin, say)
from the whole analysis; their bouts are treated as unlabelled stretches, so
nothing is counted across them.

The motif, beat-cycle and medoid machinery lives in `motifs.py` beside
this notebook; the cells here hold the knobs and the figures.

Every panel shows **decoded predictions, not ground truth**: two classes the
classifier confuses will show transitions the animal never made. Read the
figures beside the confusion matrix in `suggest_report.json`.

In [ ]:
%matplotlib inline
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

from trackanno.config import load_config
from trackanno.infer import transitions
from trackanno.infer import figures as figlib
from trackanno.infer.predict import load_predictions
from trackanno.infer.densepredict import dense_features_path

import motifs as motiflib  # notebooks/motifs.py

# ---- point this at your project ------------------------------------------- #
PROJECT = Path(os.environ.get("TRACKANNO_PROJECT", r"C:/Users/benig/powa_large/config.yaml"))

# Classes to drop from the WHOLE analysis (e.g. an "unclear" bin). Their bouts
# become stretches the decode could not label: transition pairs break across
# them instead of fabricating an A->B adjacency, neighbouring dwells are
# censored there, and the chain, motifs and features cover what remains.
EXCLUDE_CLASSES: list = ["unclear"]

cfg = load_config(PROJECT)
work = cfg.paths.work_dir

# ---- one look for every figure -------------------------------------------- #
mpl.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "figure.constrained_layout.use": True,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 9, "axes.titlesize": 10, "axes.titleweight": "normal",
    "axes.labelsize": 9, "xtick.labelsize": 8, "ytick.labelsize": 8,
    "figure.titlesize": 10, "figure.titleweight": "normal",
    "legend.fontsize": 8, "legend.title_fontsize": 8, "legend.frameon": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
})

# Okabe-Ito: colour-blind safe, print safe.
_OKABE = ["#0072B2", "#E69F00", "#009E73", "#D55E00", "#CC79A7",
          "#56B4E9", "#F0E442", "#000000"]

_SIOCO = ["#E41A1C", "#377EB8", "#4DAF4A", "#984EA3", "#FF7F00",
          "#FFFF33", "#A65628", "#F781BF", "#999999"]

_ACADIA = ["#FED789FF", "#023743FF", "#72874EFF", "#476F84FF", "#A4BED5FF", "#453947FF"]

_FIGDIR = os.environ.get("NB_SAVEFIG", "")
_saved = [0]


def show(fig, name):
    """Inline in the notebook; saved to NB_SAVEFIG when run as a script."""
    if _FIGDIR:
        fig.savefig(Path(_FIGDIR) / f"{_saved[0]:02d}_{name}.png", bbox_inches="tight")
        _saved[0] += 1
        plt.close(fig)
    else:
        plt.show()

## Load the predictions and the fitted chain

In [ ]:
bouts = load_predictions(cfg)
fps = transitions.movie_fps(cfg)
n_excluded = 0
if EXCLUDE_CLASSES:
    drop = {str(c).strip().lower() for c in EXCLUDE_CLASSES}
    for col in [c for c in bouts.columns if c.endswith("_pred")]:
        hit = bouts[col].astype(str).str.strip().str.lower().isin(drop)
        n_excluded += int(hit.sum())
        bouts.loc[hit, col] = ""
# The same call transitions.compute makes, on the (possibly blanked) table.
_mreport, _mreport_path = transitions.model_report(cfg)
report = transitions.analyse(
    bouts, fps=fps,
    min_bout_frames=int(cfg.predict.min_bout_frames),
    source=str(transitions.predictions_path(cfg)),
    model_report=_mreport, model_report_path=_mreport_path,
)

GROUP = next(iter(report["groups"]))
G = report["groups"][GROUP]
M = G["markov"]
classes = list(G["classes"])
palette = {c: _ACADIA[i % len(_ACADIA)] for i, c in enumerate(classes)}
unit = M["unit"]

sub = bouts.loc[bouts["group"].astype(str) == GROUP].copy()
sub["label"] = sub[f"{GROUP}_pred"].map(transitions.bout_state)
sub = sub.sort_values(["movie_id", "track_id", "f_start"], kind="stable")
movies = sorted(sub["movie_id"].astype(str).unique())

print(f"group {GROUP!r}: {len(sub)} bouts, classes {classes}")
if n_excluded:
    print(f"  EXCLUDE_CLASSES {EXCLUDE_CLASSES}: {n_excluded} bout(s) now "
          "count as unlabelled stretches (grey in the ethograms)")
print(f"lifetimes in {unit}; movies: {', '.join(movies)}")
for caveat in report["caveats"][:2]:
    print(f"  ! {caveat}")

## 1 - Mode switching

The transition **graph**: node size is the share of time the fitted chain
gives each state in the long run (steady state; observed occupancy when the
chain has none), edge width is P(next state | this bout ended), and the edge
label carries that probability with the rate `q` beside it. The diagonal is
empty by construction: a decode emits maximal runs.

In [ ]:
%matplotlib inline
figlib.transition_graph(M, palette, group=GROUP), "transition_graph"
plt.show()

The same numbers as a **matrix**, with the raw counts every probability came
from - a probability over three pairs is not the same evidence as one over
three hundred.

In [ ]:
show(figlib.jump_matrix(G, group=GROUP), "jump_matrix")

**Dwell-time survival.** Dots are the Kaplan-Meier estimate (right-censored
bouts folded in as lower bounds), the dashed line the constant-hazard fit
whose mean is the reported lifetime. On the log axis an exponential is a
straight line: points bending away from it mean the state does not leave at
a constant rate, and one lifetime cannot stand for it.

In [ ]:
show(figlib.survival(M, palette, group=GROUP), "survival")

Lifetimes with their track-bootstrap errors, and the model-adequacy check
the module insists on: the **steady state the fitted chain predicts** next
to the **share of time actually observed**. If the two disagree, the chain
does not describe these bouts.

In [ ]:
show(figlib.lifetimes_occupancy(M, palette, group=GROUP), "lifetimes_occupancy")

## 2 - Behavioural tracks

An **ethogram**: every track as a bar over time, coloured by the decoded
behaviour. Grey is a stretch the decode could not label. This is the raw
material every number above was computed from.

In [ ]:
def _seconds(movie_id):
    return fps.get(str(movie_id))


for movie_id in movies[:3]:
    rows = sub[sub["movie_id"].astype(str) == str(movie_id)]
    show(figlib.ethogram(rows, classes, palette, fps=_seconds(movie_id),
                         movie_id=movie_id), f"ethogram_{movie_id}")

The same bouts drawn **in space**: each track's centroid path, coloured by
the behaviour predicted along it. Image convention (y down), equal aspect.

In [ ]:
_paths_cache = {m: motiflib.track_paths(cfg, m) for m in movies}

for movie_id in movies[:2]:
    tracks = _paths_cache.get(str(movie_id))
    if tracks is None:
        print(f"{movie_id}: no tracks.parquet; skipped")
        continue
    rows = sub[sub["movie_id"].astype(str) == str(movie_id)]
    show(figlib.trajectories(tracks, rows, classes, palette,
                             movie_id=movie_id), f"trajectories_{movie_id}")

## 3 - Behavioural motifs, keypoint-MoSeq style

The trajectory-plot idiom of keypoint-MoSeq. Instance extraction follows
their `get_instance_trajectories` / `get_typical_trajectories`, and the
drawing is their `plot_trajectories`, ported:

* an **instance** is a window around a bout's ONSET (`PRE_S` before to
  `POST_S` after, kpms defaults 0.167 s / 0.5 s);
* each instance is aligned by **one rigid transform fixed at onset** -
  translated so the body anchor at onset sits at the origin, rotated so the
  heading at onset points along +x. Motion DURING the bout is kept: a
  forward swim marches right, a turn curls, stillness stays put;
* the class motif is the **element-wise `nanmedian` across instances**;
* all motifs share ONE canvas and ONE set of tile limits, so their scales
  are comparable. Skeleton edges are double-stroked (black under colour),
  nodes are outlined, and after each timestep the whole canvas is washed
  with translucent white - older poses whiten, the final pose sits
  saturated on top. A single scale bar sits at the canvas corner; `n`, the
  window and the fold split are printed above the figure.

A beat that is not phase-locked to bout onset medians toward the midline
here; its amplitude survives in the phase-averaged cycle section below.

**Folding.** A behaviour that curves to EITHER side cancels itself in the
median, so every instance is reflected onto one canonical side first
(`FOLD_LATERAL`). The price is a known bias: a straight class's symmetric
noise folds one-sided too, and its motif can carry a slight bend of about
the median |drift|. The printed side split is the guide - a clear curve
with a near 50:50 split is a genuinely two-sided behaviour; a slight bend
that is small against the travel is the folding artefact.

**The next cell is the control panel.** Exclude noisy keypoints by name or
index, fix the shared tile limits, or switch folding off.

In [ ]:
# ---- motif knobs ---------------------------------------------------------- #
# How the instance window is chosen. Bout durations here spread over an order
# of magnitude, where a mouse syllable sits near 0.4 s, so kpms's fixed window
# truncates long bouts and runs past short ones into the NEXT behaviour.
#   "auto":     per class, the window is that class's own median dwell
#               (pre-context a quarter of it); frames past each bout's real end
#               are masked, so a successor behaviour never leaks into the tail.
#   "relative": every bout time-warped onto 0..100% of its own duration -
#               shape survives, absolute speed is given up.
#   "fixed":    PRE_S/POST_S below, as kpms does it (end-masking still applies).
WINDOW_MODE = "auto"
PRE_S, POST_S = 0.167, 0.5   # used by "fixed"; kpms generate_trajectory_plots defaults
NUM_SAMPLES = 20             # common time grid, so mixed frame rates align
MIN_INSTANCES = 5            # fewer instances than this and the class is skipped
N_POSES = 20                 # skeletons drawn along the median (kpms num_timesteps)

# Keypoints to leave OUT of the drawing, by name (keypoints.nodes) or index.
# Alignment always uses Head (0) and Base (1), plotted or not.
EXCLUDE_KEYPOINTS: list = []                     # e.g. ["Head"] or [0, 7]

# Axis limits: None = automatic, or a (lo, hi) tuple. TRAJ_* set the shared
# tile limits of the motif and medoid canvases (give both to override).
TRAJ_XLIM, TRAJ_YLIM = None, None                # kpms canvas tiles
POSTURE_XLIM, POSTURE_YLIM = None, None          # phase-averaged cycle panels

# A behaviour that curves to EITHER side cancels itself in the median: the
# left- and right-curving instances are mirror images and their element-wise
# median is a straight smear. (kpms rarely shows this because its unsupervised
# syllables usually split the two chiralities into separate syllables; a human
# label lumps them.) Folding reflects every instance about the forward axis
# onto one canonical side before the median, and the printed note reports the
# side split. The price is a known bias: a STRAIGHT class's symmetric lateral
# noise is folded one-sided too, so its motif can carry a slight bend of about
# the median |drift|. Read a bend that is small against the travel as that
# artefact, not as a turn; there is no data-driven test that separates the two
# for every behaviour, so the fold is constant and the bias documented.
FOLD_LATERAL = True

K = cfg.n_keypoints
NODE_NAMES = list(getattr(cfg.keypoints, "nodes", None) or
                  [f"kp{k}" for k in range(K)])
_excluded = set()
for entry in EXCLUDE_KEYPOINTS:
    if isinstance(entry, (int, np.integer)) and 0 <= int(entry) < K:
        _excluded.add(int(entry))
    elif str(entry) in NODE_NAMES:
        _excluded.add(NODE_NAMES.index(str(entry)))
    else:
        print(f"EXCLUDE_KEYPOINTS entry {entry!r} matches no keypoint; "
              f"names are {NODE_NAMES}")
# The physical chain: Head, Base, then the flagellum in order.
CHAIN = [k for k in range(K) if k not in _excluded]
FLAG_CHAIN = [k for k in CHAIN if k >= 1]
if _excluded:
    print("excluded from the drawing:", sorted(NODE_NAMES[k] for k in _excluded))

In [ ]:
CHAIN

In [ ]:
_NODE_COLOURS = plt.get_cmap("autumn")(np.linspace(0, 1, K))  # kpms default


def class_instances(name):
    """Onset-aligned instances of one class, under the knobs above."""
    return motiflib.class_instances(
        sub[sub["label"] == name], _paths_cache.get, _seconds,
        n_keypoints=K, window_mode=WINDOW_MODE, pre_s=PRE_S, post_s=POST_S,
        num_samples=NUM_SAMPLES)


def motifs():
    if K < 4:
        print("not a pose project; motif panels need keypoints")
        return None
    import warnings as _w
    titles, medians = [], []
    for name in classes:
        instances, imeta = class_instances(name)
        if instances is None or len(instances) < MIN_INSTANCES:
            n = 0 if instances is None else len(instances)
            print(f"{name}: only {n} usable instance(s); skipped")
            continue
        onset_at = int(round((NUM_SAMPLES - 1) * imeta["onset_frac"]))
        instances, fold_note = motiflib.fold_instances(
            instances, onset_at, enabled=bool(FOLD_LATERAL))
        with _w.catch_warnings():
            # All-NaN timepoints are expected: pre-onset frames can predate
            # the track itself.
            _w.simplefilter("ignore", RuntimeWarning)
            medians.append(np.nanmedian(instances, axis=0))
        titles.append(name)
        print(f"{name}: {imeta['note']}, n={len(instances)}{fold_note}")
    if not medians:
        print("no class has enough instances for a motif")
        return None
    picks = np.linspace(0, NUM_SAMPLES - 1, N_POSES).round().astype(int)
    lims = (np.array([[TRAJ_XLIM[0], TRAJ_YLIM[0]], [TRAJ_XLIM[1], TRAJ_YLIM[1]]])
            if TRAJ_XLIM and TRAJ_YLIM else None)
    return motiflib.plot_trajectories(
        titles, [median[picks][:, CHAIN] for median in medians],
        colors=_NODE_COLOURS[CHAIN], lims=lims)


fig = motifs()
if fig is not None:
    show(fig, "motifs")

**Usage**, the other MoSeq staple: how often each behaviour occurs, how much
of the record it owns, and how long a bout of it typically lasts.

In [ ]:
show(figlib.usage(sub["label"], G, palette, group=GROUP), "usage")

## 4 - Feature distributions inside each behaviour

Each dense window is given the behaviour decoded at its centre frame, and
the feature the pipeline measured for that window is split by class. This is
a sanity check as much as a result: if "immotile" does not sit at low speed,
the model or the decode is doing something worth investigating.

The speed and turn model features average |per-frame steps| of the BOX
centre: the absolute value rectifies localisation jitter into a floor of
noise x fps, and a sweeping flagellum drags the box edge, so a stationary
flapper reads ~1000 px/s (`speed_mean`) and hundreds of rad/s
(`turn_abs_mean`). Fine as classifier inputs, misleading as physical rates,
so the panels below show the net-over-window speed and the body-anchor
speed instead; `body_rotation_speed` keeps a milder version of the same
floor and is best read as relative between classes.

In [ ]:
# What to draw, and honestly. speed_mean and turn_abs_mean are model FEATURES,
# not physical rates: both average |per-frame step| of the BOX centre, which
# rectifies localisation jitter into a floor of noise x fps and reads a
# flagellar sweep as locomotion (a stationary flapper shows ~1000 px/s and
# hundreds of rad/s). net_speed is displacement over the whole window, which
# cancels both. body_rotation_speed is the head-base axis rate: part real
# beat-driven wobble, part the same rectified jitter floor; read it as
# relative between classes, not as revolutions per second.
FEATURES_TO_SHOW = [
    ("net_speed", "net speed (px/s)"),
    ("body_speed_mean", "body-anchor speed (px/s)"),
    ("straightness", "straightness"),
    ("wave_freq_hz", "beat frequency (Hz)"),
    ("wave_amp_mean", "beat amplitude (rad)"),
    ("body_rotation_speed", "axis rotation (rad/s)"),
]


def _features_with_labels():
    path = dense_features_path(cfg)
    if not path.exists():
        path = work / "clip_features.parquet"
    if not path.exists():
        print("no feature table; run the labelling pass (or build features) first")
        return None
    feats = pd.read_parquet(path)
    if "skip_reason" in feats.columns:
        feats = feats[feats["skip_reason"].astype(str) == ""]
    if "f_end" not in feats.columns:
        feats = feats.assign(f_end=feats["f_start"] + int(cfg.clips.frames))
    feats = motiflib.label_windows(feats, sub)
    if "net_displacement" in feats.columns:
        rate = feats["movie_id"].astype(str).map(fps)
        feats = feats.assign(net_speed=feats["net_displacement"] * rate
                             / (feats["f_end"] - feats["f_start"]))
    return feats


feats = _features_with_labels()
if feats is not None and len(feats):
    fig = figlib.feature_violins(feats, classes, palette, FEATURES_TO_SHOW)
    if fig is not None:
        show(fig, "feature_violins")
else:
    print("no windows with a decoded behaviour at their centre")

And the classic **beat map**: frequency against amplitude, one dot per
window, coloured by behaviour. Clusters here that cut across the class
colours are structure the current labels do not capture: candidates for a
finer class list.

In [ ]:
if feats is not None and len(feats):
    fig = figlib.beat_map(feats, classes, palette)
    if fig is not None:
        show(fig, "beat_map")

## 5 - Motifs for oscillatory behaviour: the average beat CYCLE, and the
most typical bout

The kpms trajectory plot in section 3 averages instances at the same **time
from onset**. That is right for event-like syllables (a rear, a dart) whose
onset is the movement's own start. A swimming bout is a sustained
**oscillatory regime**: its beat is not phase-locked to the bout boundary,
so at any fixed time-from-onset the instances sit at random beat phases and
every average cancels the oscillation by construction.

Two representations that do not have that problem, one figure each:

* **the phase-averaged beat cycle** - each frame is assigned its position
  in the beat via the Hilbert phase of a reference keypoint's lateral
  motion, and the pose is averaged **per phase bin**; the standard move in
  the flagellar literature.

  Binning a trace by its own Hilbert phase rebuilds a cycle out of
  ANYTHING, tracking jitter included: sorting frames by phase is sorting
  them by lateral position, so even a frozen cell draws a "beat" the size
  of its noise. And that jitter is RED, not white: detection errors and
  slow wobble are autocorrelated, piling their power onto low frequencies,
  which a shuffle test cannot see. The oscillation gate therefore tests
  each bout against AR(1) surrogates matched to the trace's own
  autocorrelation: the strongest peak above that red background (at least
  two full cycles in the bout) must beat the surrogates' 99th percentile
  (p < 0.01) at an amplitude of at least `MIN_OSC_PX`. A class with
  nothing oscillating says so instead of showing its jitter.
* **the medoid bout** - no averaging at all: the window closest to the
  class's median vector in the CLASSIFIER'S OWN feature space (robust
  z-scores; the cnn identity block excluded, since it would elect the most
  average-looking animal rather than behaviour) - the class's most ORDINARY
  example, drawn as it happened on the same canvas style as section 3, its
  movie/track/frames printed above the figure so it can be opened in the
  GUI. Confidence is deliberately not the criterion (the most confident
  window is the most extreme one), but `MEDOID_MIN_CONF` uses it as a gate
  against ambiguous candidates.

In [ ]:
N_PHASE = 12                 # poses per beat cycle
PHASE_NODE = None            # keypoint whose lateral motion defines the phase;
                             # None = the mid-flagellum node
MIN_CYCLE_FRAMES = 12        # shorter bouts cannot support a phase estimate
MIN_OSC_PX = 5             # amplitude at the dominant frequency below this
                             # = nothing oscillating
N_NULL_SHUFFLES = 200        # AR(1) surrogates behind the oscillation test

# The medoid's feature space. "model" is every column the classifier reads
# except the cnn identity block (those fingerprint INDIVIDUALS, so a medoid
# over them drifts toward the most-photographed animal); "shown" is just the
# violin subset. The classifier's POSTERIOR is deliberately not the selection
# criterion - confidence measures distance from the decision boundary, and the
# most confident window is the most extreme example, not the most typical -
# but it works as a GATE: with MEDOID_MIN_CONF > 0 only windows given at least
# that probability of the class are candidates. "model+cnn" keeps the cnn
# block in: worth trying only to TEST whether identity is steering the choice
# - if the elected window changes a lot, it was.
MEDOID_FEATURES = "model"
MEDOID_MIN_CONF = 0.5
# Which candidate to draw: 0 = the medoid itself, 1 = the runner-up, and so on
# - a cheap way to browse the typical examples. MEDOID_OVERRIDE pins a window
# by hand per class and wins over everything else:
#   {"swim": ("movie_03", 12, 4200)}  = (movie_id, track_id, a frame inside it)
# - the coordinates printed above the figure.
MEDOID_PICK = 0
MEDOID_OVERRIDE: dict = {}
# How much footage the medoid panel draws. None = the elected window's own
# length; a number = that many SECONDS, centred on the window and clipped to
# the track. Longer than the window's own bout shows the neighbouring
# behaviour too - context, clearly visible as such since this panel is one
# real recording, never an average.
MEDOID_SECONDS = None

_ref_node = (PHASE_NODE if PHASE_NODE is not None
             else FLAG_CHAIN[len(FLAG_CHAIN) // 2])


def phase_cycle(name):
    """One class's phase-averaged cycle, under the knobs above."""
    rows = sub[(sub["label"] == name) & (sub["n_frames"] >= MIN_CYCLE_FRAMES)]
    return motiflib.phase_cycle(
        rows, _paths_cache.get, n_keypoints=K, ref_node=_ref_node,
        flag_chain=FLAG_CHAIN, n_phase=N_PHASE,
        min_cycle_frames=MIN_CYCLE_FRAMES, min_osc_px=MIN_OSC_PX,
        n_null=N_NULL_SHUFFLES, fold=bool(FOLD_LATERAL))


def medoid_window(name):
    """The class window to draw: pinned, else the medoid of its features."""
    if name in MEDOID_OVERRIDE:
        movie, track, frame = MEDOID_OVERRIDE[name]
        hit = sub[(sub["movie_id"].astype(str) == str(movie))
                  & (sub["track_id"] == int(track))
                  & (sub["f_start"] <= int(frame)) & (sub["f_end"] > int(frame))]
        if len(hit):
            row = dict(hit.iloc[0])
            if row.get("label") != name:
                print(f"  note: the pinned window for {name!r} is decoded as "
                      f"{row.get('label')!r}")
            row["_how"] = "pinned"
            return row
        print(f"  MEDOID_OVERRIDE for {name!r} matches no bout; using the medoid")
    if feats is None or not len(feats):
        return None
    rows = feats[feats["_behaviour"] == name]
    if MEDOID_MIN_CONF > 0.0 and len(rows) >= 3:
        posterior = motiflib.class_posterior(rows, name, cfg, GROUP)
        if posterior is not None and (posterior >= MEDOID_MIN_CONF).sum() >= 3:
            rows = rows[posterior >= MEDOID_MIN_CONF]
    return motiflib.elect_medoid(
        rows, space=MEDOID_FEATURES,
        shown=tuple(c for c, _ in FEATURES_TO_SHOW), pick=MEDOID_PICK)


def phase_figure():
    """The phase-averaged beat cycle, one panel per class."""
    if K < 4:
        print("not a pose project; skipped")
        return None
    results = {name: phase_cycle(name) for name in classes}
    return figlib.phase_cycles(results, classes, FLAG_CHAIN, n_phase=N_PHASE,
                               fold=bool(FOLD_LATERAL),
                               xlim=POSTURE_XLIM, ylim=POSTURE_YLIM)


def medoid_figure():
    """Each class's most ordinary bout, on the same canvas style as section 3."""
    if K < 4:
        print("not a pose project; skipped")
        return None
    titles, seqs = [], []
    for name in classes:
        r = medoid_window(name)
        if r is None:
            pool = sub[sub["label"] == name]
            r = (pool.iloc[(pool["n_frames"]
                            - pool["n_frames"].median()).abs().argmin()]
                 if len(pool) else None)
        if r is None:
            print(f"{name}: no bout to draw; skipped")
            continue
        t = _paths_cache.get(str(r["movie_id"]))
        rate = _seconds(r["movie_id"])
        f0, f1 = int(r["f_start"]), int(r["f_end"])
        if MEDOID_SECONDS and rate:
            want = max(3, int(round(float(MEDOID_SECONDS) * rate)))
            centre = (f0 + f1) // 2
            f0 = centre - want // 2
            f1 = f0 + want
        span = f1 - f0
        win = (motiflib.keypoint_window(t[t["track_id"] == r["track_id"]],
                                        f0, span, K)
               if t is not None else None)
        aligned = motiflib.align_at_onset(win, 0) if win is not None else None
        if aligned is None:
            print(f"{name}: medoid window has no usable pose; skipped")
            continue
        picks = np.linspace(0, span - 1, N_POSES).round().astype(int)
        titles.append(name)
        seqs.append(aligned[picks][:, CHAIN])
        length = span / rate if rate else span
        print(f"{name}: {r.get('_how') or 'medoid'} window, {r['movie_id']} · "
              f"track {int(r['track_id'])} · f{f0}-{f1} "
              f"({length:.2g}{'s' if rate else ' frames'})")
    if not seqs:
        print("no class has a drawable medoid")
        return None
    return motiflib.plot_trajectories(titles, seqs,
                                      colors=_NODE_COLOURS[CHAIN])


fig = phase_figure()
if fig is not None:
    show(fig, "phase_cycle")
fig = medoid_figure()
if fig is not None:
    show(fig, "medoid")

---
**Where to go from here**

* The numbers behind section 1 live in `transitions.json`
  (`trackanno transitions --config ...` prints them with
  their caveats); every error bar is a bootstrap over whole tracks.
* `fit_max_deviation` above ~0.1 in the survival panel means a state is not
  exponential: consider whether it is really two behaviours worth labelling
  apart.
* Everything here inherits the classifier's errors. `suggest_report.json`
  carries the grouped-CV confusion matrix; read the jump matrix beside it,
  because a confused class pair produces transitions the animal never made.